# Carbon MAS v2 — Análise dos resultados do sistema (`_manifest.json`)

Notebook de apoio à dissertação. Carrega o manifest produzido por `run_system_tests.py`
e produz:

1. **Visão geral** do lote (sessões, ok/erros, datasets, tempo).
2. **Análise de acerto do oráculo agêntico** — matriz de confusão entre o resultado
   *esperado* de cada cenário sintético e a *decisão de governança* obtida, com:
   - **Falsos negativos**: dados que *deveriam* gerar crédito e **não** geraram.
   - **Falsos positivos**: dados que **não** deveriam gerar crédito mas **geraram** (fraude que passou).
3. **Economia de CO₂** por viagem (gráfico).
4. **Emissões vs distância** (g/km) e tipos de anomalia.
5. **Estatísticas** de confiança e tempo de execução.


In [ ]:
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_colwidth', 60)
pd.set_option('display.width', 160)

# ── Configuração ────────────────────────────────────────────────────────────
RESULTS_DIR = Path('results')        # pasta raiz dos runs
MANIFEST_PATH = None                 

def find_latest_manifest(results_dir: Path) -> Path:
    manifests = sorted(results_dir.glob('run_*/_manifest.json'))
    if not manifests:
        raise FileNotFoundError(f'Nenhum _manifest.json encontrado em {results_dir}/run_*/')
    return manifests[-1]

manifest_path = Path(MANIFEST_PATH) if MANIFEST_PATH else find_latest_manifest(RESULTS_DIR)
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
print('Manifest :', manifest_path)
print('Run ID   :', manifest.get('run_id'))
print('Data dir :', manifest.get('data_dir'))
print('Dry-run  :', manifest.get('dry_run'))
print(f"Sessões  : {manifest.get('total')}  (ok={manifest.get('ok')}, erros={manifest.get('errors')})")

In [ ]:
# Sessões → DataFrame
df = pd.DataFrame(manifest['sessions'])

# Tipos numéricos (campos podem faltar em sessões com erro)
for col in ['total_co2_g', 'co2_saved_g', 'distance_km', 'confidence', 'credits_cct', 'elapsed_s']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

is_synthetic = (df.get('dataset') == 'synthetic').any() if 'dataset' in df.columns else False
print(f"Layout detectado: {'SINTÉTICO (com gabarito)' if is_synthetic else 'REAL (sem gabarito)'}")
df.head(12)

## 1. Visão geral do lote

In [ ]:
print('Status das sessões:')
print(df['status'].value_counts().to_string())

if 'governance_decision' in df.columns:
    print('\nDecisão de governança:')
    print(df['governance_decision'].fillna('(erro)').value_counts().to_string())

if 'total_co2_g' in df.columns:
    print('\nResumo numérico (sessões ok):')
    cols = [c for c in ['total_co2_g', 'co2_saved_g', 'distance_km', 'credits_cct', 'confidence', 'elapsed_s'] if c in df.columns]
    display(df[df['status'] == 'ok'][cols].describe().round(2))

## 2. Análise de acerto do oráculo agêntico

Para cada cenário sintético, o **resultado esperado** (deve gerar crédito ou não) é
conhecido a partir do gerador de CSVs. Comparamos com a **decisão de governança** real.

| Cenário (`rodada`) | Esperado | Motivo |
|---|---|---|
| 01 viagem perfeita | gerar crédito | controle/ideal |
| 02 eco-motorista | gerar crédito | alta economia |
| 03 viagem etanol | gerar crédito | física estequiométrica válida |
| 04 falha sensor MAF (válido) | gerar crédito | fallback Lei dos Gases deve aprovar |
| 05 motorista agressivo | negar | emite demais (0 g salvos) |
| 06 velocidade impossível | negar | fraude de GPS |
| 07 motor desligado em movimento | negar | fraude física |
| 08 bot gerador de dados | negar | padrão robótico |
| 09 sensor de temperatura hackeado | negar | anomalia de sensor |
| 10 arquivo vazio/corrompido | negar | resiliência a erro |

In [ ]:
# Gabarito por número de cenário (rodada). True = deveria gerar crédito.
EXPECTED_ISSUE = {
    '01': True,  '02': True,  '03': True,  '04': True,
    '05': False, '06': False, '07': False, '08': False, '09': False, '10': False,
}

def expected_label(row):
    key = str(row.get('rodada', '')).zfill(2)
    exp = EXPECTED_ISSUE.get(key)
    if exp is None:
        return None
    return 'gerar_credito' if exp else 'negar'

def actual_label(row):
    if row.get('status') != 'ok':
        return 'negar'  # erro/corrompido = nenhum crédito emitido
    return 'gerar_credito' if row.get('governance_decision') == 'approved' else 'negar'

if is_synthetic:
    df['esperado'] = df.apply(expected_label, axis=1)
    df['obtido'] = df.apply(actual_label, axis=1)
    df['acerto'] = df['esperado'] == df['obtido']
    graded = df[df['esperado'].notna()].copy()
    acc = graded['acerto'].mean() if len(graded) else float('nan')
    print(f'Sessões avaliadas: {len(graded)}   |   Acurácia do sistema: {acc:.1%}')
else:
    graded = pd.DataFrame()
    print('Dados reais sem gabarito — análise de acerto ignorada.')

In [ ]:
# Matriz de confusão (esperado x obtido)
if is_synthetic and len(graded):
    cm = pd.crosstab(graded['esperado'], graded['obtido'],
                     rownames=['Esperado'], colnames=['Obtido'], dropna=False)
    display(cm)

    tp = int(((graded['esperado'] == 'gerar_credito') & (graded['obtido'] == 'gerar_credito')).sum())
    tn = int(((graded['esperado'] == 'negar') & (graded['obtido'] == 'negar')).sum())
    fp = int(((graded['esperado'] == 'negar') & (graded['obtido'] == 'gerar_credito')).sum())
    fn = int(((graded['esperado'] == 'gerar_credito') & (graded['obtido'] == 'negar')).sum())
    print(f'\nVerdadeiros positivos (crédito correto) : {tp}')
    print(f'Verdadeiros negativos (negado correto)  : {tn}')
    print(f'FALSOS positivos (fraude que passou)    : {fp}')
    print(f'FALSOS negativos (legítimo negado)      : {fn}')
    if (tp + fp):
        print(f'\nPrecisão (créditos emitidos corretos)   : {tp / (tp + fp):.1%}')
    if (tp + fn):
        print(f'Recall   (legítimos efetivamente pagos) : {tp / (tp + fn):.1%}')

In [ ]:
# CASO DE SUCESSO / FALHA — listas detalhadas para a dissertação
show_cols = [c for c in ['vehicle', 'viagem', 'scenario', 'total_co2_g', 'co2_saved_g',
                         'distance_km', 'confidence', 'anomaly', 'governance_decision',
                         'credits_cct'] if c in df.columns]

if is_synthetic and len(graded):
    fp_rows = graded[(graded['esperado'] == 'negar') & (graded['obtido'] == 'gerar_credito')]
    fn_rows = graded[(graded['esperado'] == 'gerar_credito') & (graded['obtido'] == 'negar')]

    print('=== FALSOS POSITIVOS — fraudes/inválidos que GERARAM crédito (não deveriam) ===')
    display(fp_rows[show_cols] if len(fp_rows) else 'Nenhum — o sistema bloqueou todas as fraudes. ✅')

    print('\n=== FALSOS NEGATIVOS — viagens legítimas que NÃO geraram crédito (deveriam) ===')
    display(fn_rows[show_cols] if len(fn_rows) else 'Nenhum — todas as viagens válidas foram premiadas. ✅')

## 3. Economia de CO₂ por viagem

In [ ]:
ok = df[df['status'] == 'ok'].copy()
if 'co2_saved_g' in ok.columns and len(ok):
    label_col = 'scenario' if 'scenario' in ok.columns else 'vehicle_id'
    plot_df = ok.sort_values('co2_saved_g')
    colors = ['#2ca02c' if v > 0 else '#d62728' for v in plot_df['co2_saved_g']]

    fig, ax = plt.subplots(figsize=(10, max(3, 0.4 * len(plot_df))))
    ax.barh(plot_df[label_col].astype(str), plot_df['co2_saved_g'], color=colors)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_xlabel('Economia de CO₂ vs baseline (g)   — verde = crédito, vermelho = sem economia')
    ax.set_title('Economia de CO₂ por viagem')
    plt.tight_layout()
    plt.show()
else:
    print('Sem co2_saved_g para plotar.')

## 4. Emissões, distância e anomalias

In [ ]:
if {'distance_km', 'total_co2_g'}.issubset(ok.columns) and len(ok):
    d = ok[ok['distance_km'] > 0].copy()
    d['g_por_km'] = d['total_co2_g'] / d['distance_km']

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    axes[0].scatter(d['distance_km'], d['total_co2_g'], c='#1f77b4', alpha=0.8)
    axes[0].set_xlabel('Distância (km)'); axes[0].set_ylabel('CO₂ total (g)')
    axes[0].set_title('CO₂ total vs distância')

    axes[1].axhline(180, color='gray', linestyle='--', label='baseline 180 g/km (IPCC)')
    axes[1].bar(range(len(d)), d['g_por_km'].values, color='#ff7f0e')
    axes[1].set_xticks(range(len(d)))
    lbl = d['scenario'] if 'scenario' in d.columns else d['vehicle_id']
    axes[1].set_xticklabels(lbl.astype(str), rotation=90, fontsize=7)
    axes[1].set_ylabel('g CO₂ / km'); axes[1].set_title('Intensidade de emissão por viagem')
    axes[1].legend()
    plt.tight_layout(); plt.show()

    display(d[['scenario', 'distance_km', 'total_co2_g', 'g_por_km']].round(1)
            if 'scenario' in d.columns else d[['distance_km', 'total_co2_g', 'g_por_km']].round(1))

In [ ]:
if 'anomaly' in df.columns:
    counts = df['anomaly'].fillna('(erro)').value_counts()
    fig, ax = plt.subplots(figsize=(8, 4))
    counts.plot(kind='bar', ax=ax, color='#9467bd')
    ax.set_ylabel('nº de sessões'); ax.set_title('Tipos de anomalia detectados')
    plt.tight_layout(); plt.show()
    print(counts.to_string())

## 5. Confiança e tempo de execução

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
if 'confidence' in ok.columns and ok['confidence'].notna().any():
    axes[0].hist(ok['confidence'].dropna(), bins=10, color='#17becf', edgecolor='black')
    axes[0].set_xlabel('Confiança (0–100)'); axes[0].set_ylabel('nº de sessões')
    axes[0].set_title('Distribuição do score de confiança')
if 'elapsed_s' in ok.columns and ok['elapsed_s'].notna().any():
    axes[1].hist(ok['elapsed_s'].dropna(), bins=10, color='#bcbd22', edgecolor='black')
    axes[1].set_xlabel('Tempo por sessão (s)'); axes[1].set_ylabel('nº de sessões')
    axes[1].set_title(f"Tempo de pipeline (média {ok['elapsed_s'].mean():.1f}s)")
plt.tight_layout(); plt.show()

## 6. Exportar tabela consolidada

Salva um CSV ao lado do manifest para colar na dissertação / abrir no Excel.

In [ ]:
out_csv = manifest_path.parent / 'analise_consolidada.csv'
df.to_csv(out_csv, index=False, encoding='utf-8-sig')
print('Tabela exportada →', out_csv)